# 🚀 ProFoods × UnEmplacement.com — Scraping & Matching Production

Ce notebook pilote :
1. **Cellule 1** : Installation & imports.
2. **Cellule 2** : Configuration & sécurisation des constantes.
3. **Cellule 3** : Engine de scraping haute-disponibilité asynchrone avec Playwright.
4. **Cellule 4** : Intégration CSV ProFoods + algorithme de matching.
5. **Cellule 5** : Export local + schéma SQL Vercel/Supabase.

## CELLULE 1 : INSTALLATION & IMPORTS

In [ ]:
# --- CELLULE 1 : INSTALLATION & IMPORTS ---

# Installation silencieuse des dépendances
!pip install -q playwright pandas openpyxl

# Téléchargement des navigateurs Playwright (Chromium par défaut)
!playwright install chromium

# Imports standards
import asyncio
import os
import re
import json
import random
import time
import datetime as dt
from typing import List, Dict, Any, Optional
from urllib.parse import urljoin, urlparse

import pandas as pd
from playwright.async_api import async_playwright, Page, Browser, BrowserContext, Response

print(f"[OK] Environnement prêt — {dt.datetime.now().isoformat()}")

## CELLULE 2 : CONFIGURATION & SÉCURISATION DES CONSTANTES

In [ ]:
# --- CELLULE 2 : CONFIGURATION & SÉCURISATION DES CONSTANTES ---

# Préférez les variables d'environnement pour ne jamais commiter de credentials.
# Exemple : export UNEMPLACEMENT_EMAIL="ton@email.com" dans votre shell.
CONFIG = {
    "UNEMPLACEMENT_EMAIL": os.environ.get("UNEMPLACEMENT_EMAIL", ""),
    "UNEMPLACEMENT_PASSWORD": os.environ.get("UNEMPLACEMENT_PASSWORD", ""),
    "PATH_CSV_PROFOODS": os.environ.get("PATH_CSV_PROFOODS", "./data/biens_profoods.csv"),
    "URL_LOGIN": "https://www.unemplacement.com/connexion",
    "URL_LEADS": "https://www.unemplacement.com/demandes-locaux",  # À adapter après inspection du site
    "HEADLESS": os.environ.get("PLAYWRIGHT_HEADLESS", "true").lower() == "true",
    "SLOW_MO": int(os.environ.get("PLAYWRIGHT_SLOW_MO", "150"))
}

if not CONFIG["UNEMPLACEMENT_EMAIL"] or not CONFIG["UNEMPLACEMENT_PASSWORD"]:
    raise ValueError(
        "Credentials UnEmplacement.com manquants. "
        "Exportez UNEMPLACEMENT_EMAIL et UNEMPLACEMENT_PASSWORD avant d'exécuter ce notebook."
    )

# Rotation de User-Agents récents (Chrome / Windows, macOS, Linux)
USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36 Edg/123.0.0.0"
]

# --- Sélecteurs HTML de fallback (modifiables facilement) ---
SELECTEUR_CARTE_ANNONCE = "div[class*='card'], article[class*='annonce'], [data-testid*='annonce']"
SELECTEUR_TITRE = "h2, h3, .title, [class*='title'], [class*='titre']"
SELECTEUR_DESCRIPTION = ".description, [class*='description'], [class*='desc'], p"
SELECTEUR_LOCALISATION = ".location, [class*='location'], [class*='ville'], [class*='adresse']"
SELECTEUR_SURFACE = ".surface, [class*='surface'], [class*='m2'], [class*='m²']"
SELECTEUR_BUDGET = ".budget, [class*='budget'], [class*='prix'], [class*='loyer']"
SELECTEUR_LIEN = "a[href]"
SELECTEUR_BOUTON_PAGINATION = "button[class*='pagination'], a[class*='pagination'], [class*='next'], [class*='suivant']"
SELECTEUR_BOUTON_CHARGER_PLUS = "button[class*='load'], button[class*='more'], [class*='charger-plus'], [class*='voir-plus']"

# Mots-clés de détection du besoin extraction / cuisine professionnelle
MOTS_CLEFS_EXTRACTION = [
    "hotte", "extraction", "déjection", "ventilation",
    "cloud kitchen", "dark kitchen", "ghost kitchen",
    "restaurant", "traiteur", "snack", "fast-food", "boulangerie", "pâtisserie"
]

print("[OK] Configuration et sélecteurs initialisés.")

## CELLULE 3 : LE ENGINE DE SCRAPING HAUTE-DISPONIBILITÉ (ASYNCHRONE)

In [ ]:
# --- CELLULE 3 : ENGINE DE SCRAPING HAUTE-DISPONIBILITÉ ---

def _human_delay(min_ms: float = 80, max_ms: float = 350) -> float:
    """Retourne un délai aléatoire pour simuler la frappe humaine."""
    return random.uniform(min_ms, max_ms) / 1000.0


async def _human_type(page: Page, selector: str, text: str) -> None:
    """Saisie caractère par caractère avec délai variable."""
    await page.click(selector)
    await page.fill(selector, "")
    for char in text:
        await page.type(selector, char, delay=random.uniform(50, 250))
        await asyncio.sleep(_human_delay(30, 120))


async def _human_mouse_move(page: Page, n_moves: int = 3) -> None:
    """Déplace la souris de manière erratique pour simuler un humain."""
    viewport = await page.evaluate("() => ({ w: window.innerWidth, h: window.innerHeight })")
    for _ in range(n_moves):
        x = random.randint(50, viewport["w"] - 50)
        y = random.randint(50, viewport["h"] - 50)
        await page.mouse.move(x, y)
        await asyncio.sleep(_human_delay(150, 500))


async def _patch_webdriver(page: Page) -> None:
    """Masque navigator.webdriver et plugins pour éviter la détection automatisée."""
    await page.add_init_script("""
        Object.defineProperty(navigator, 'webdriver', { get: () => undefined });
        Object.defineProperty(navigator, 'plugins', { get: () => [1, 2, 3, 4, 5] });
        window.chrome = { runtime: {} };
        Object.defineProperty(window, 'chrome', { get: () => ({ runtime: {} }) });
    """)


def _extraire_nombre(texte: str) -> Optional[float]:
    """Extrait le premier nombre entier/flottant d'une chaîne (surface, budget)."""
    if not texte:
        return None
    texte = texte.replace("\u202f", "").replace(" ", "").replace(",", ".")
    match = re.search(r"\d+(?:\.\d+)?", texte)
    return float(match.group(0)) if match else None


def _detecter_besoin_extraction(texte: str) -> bool:
    """Détecte si le texte mentionne un besoin d'extraction/cuisine pro."""
    if not texte:
        return False
    texte_lower = texte.lower()
    return any(mot in texte_lower for mot in MOTS_CLEFS_EXTRACTION)


def _normaliser_ville(ville: str) -> str:
    """Nettoie et met en majuscules la ville."""
    if not ville:
        return ""
    return re.sub(r"\s+", " ", ville).strip().upper()


def _normaliser_code_postal(cp: str) -> str:
    """Retourne un code postal à 5 chiffres ou une chaîne vide."""
    if not cp:
        return ""
    digits = re.sub(r"\D", "", cp)
    return digits[:5] if len(digits) >= 5 else digits


async def _extraire_annonces_json(captured_responses: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    Tente de normaliser les payloads JSON interceptés en annonces UnEmplacement.
    Adaptez les clés selon le vrai schema de l'API du site.
    """
    annonces = []
    for payload in captured_responses:
        try:
            data = payload if isinstance(payload, dict) else json.loads(payload)
            # Si la réponse contient une liste d'annonces
            candidates = data if isinstance(data, list) else data.get("data", data.get("annonces", data.get("results", [])))
            if not isinstance(candidates, list):
                candidates = [candidates]
            for item in candidates:
                if not isinstance(item, dict):
                    continue
                annonces.append({
                    "id_annonce": str(item.get("id", item.get("_id", item.get("slug", f"json_{len(annonces)}")))),
                    "ville": _normaliser_ville(str(item.get("ville", item.get("city", item.get("localisation", ""))))),
                    "code_postal": _normaliser_code_postal(str(item.get("code_postal", item.get("zip", item.get("cp", ""))))),
                    "surface_min": int(_extraire_nombre(str(item.get("surface_min", item.get("min_surface", 0)))) or 0),
                    "surface_max": int(_extraire_nombre(str(item.get("surface_max", item.get("max_surface", item.get("surface", 0))))) or 0),
                    "budget_max_mensuel": float(_extraire_nombre(str(item.get("budget", item.get("budget_max", item.get("loyer_max", 0))))) or 0.0),
                    "activite_detaillee": str(item.get("description", item.get("activite", item.get("titre", "")))).strip(),
                    "besoin_extraction": _detecter_besoin_extraction(str(item.get("description", item.get("activite", "")))),
                    "url_contact": str(item.get("url", item.get("lien", item.get("slug", "")))).strip(),
                })
        except Exception as e:
            print(f"[WARN] Échec parsing payload JSON : {e}")
    return annonces


async def _extraire_annonces_html(page: Page) -> List[Dict[str, Any]]:
    """Fallback HTML : parse les cartes d'annonces avec les sélecteurs configurés."""
    annonces = []
    cartes = await page.locator(SELECTEUR_CARTE_ANNONCE).all()
    print(f"[INFO] {len(cartes)} cartes HTML détectées.")

    for idx, carte in enumerate(cartes):
        try:
            titre = await carte.locator(SELECTEUR_TITRE).first.text_content() or ""
            description = await carte.locator(SELECTEUR_DESCRIPTION).first.text_content() or ""
            localisation = await carte.locator(SELECTEUR_LOCALISATION).first.text_content() or ""
            surface_text = await carte.locator(SELECTEUR_SURFACE).first.text_content() or ""
            budget_text = await carte.locator(SELECTEUR_BUDGET).first.text_content() or ""

            # Extraction ville / CP depuis la localisation
            cp_match = re.search(r"\b\d{5}\b", localisation)
            code_postal = cp_match.group(0) if cp_match else ""
            ville = re.sub(r"\b\d{5}\b", "", localisation).strip(",-").strip()

            # Surfaces : recherche de deux nombres (min/max) ou d'un seul
            nombres_surface = [float(x) for x in re.findall(r"\d+(?:\.\d+)?", surface_text)]
            surface_min = int(nombres_surface[0]) if nombres_surface else 0
            surface_max = int(nombres_surface[1]) if len(nombres_surface) > 1 else surface_min

            # Budget
            budget_max = _extraire_nombre(budget_text) or 0.0

            # Lien de contact
            lien = ""
            lien_el = await carte.locator(SELECTEUR_LIEN).first
            if lien_el:
                href = await lien_el.get_attribute("href") or ""
                lien = urljoin(CONFIG["URL_LEADS"], href)

            texte_complet = f"{titre} {description} {localisation}".strip()

            annonces.append({
                "id_annonce": f"html_{idx}",
                "ville": _normaliser_ville(ville),
                "code_postal": _normaliser_code_postal(code_postal),
                "surface_min": surface_min,
                "surface_max": surface_max,
                "budget_max_mensuel": budget_max,
                "activite_detaillee": texte_complet[:500],
                "besoin_extraction": _detecter_besoin_extraction(texte_complet),
                "url_contact": lien
            })
        except Exception as e:
            print(f"[WARN] Carte {idx} ignorée : {e}")
            continue
    return annonces


async def _login(page: Page) -> None:
    """Connexion à UnEmplacement.com avec comportement humain."""
    print(f"[INFO] Connexion à {CONFIG['URL_LOGIN']}")
    await page.goto(CONFIG["URL_LOGIN"], wait_until="networkidle")
    await _human_mouse_move(page, n_moves=2)

    # Détection des champs email/password par attributs génériques
    email_selectors = ["input[type='email']", "input[name='email']", "input[id*='email']", "input[placeholder*='email']"]
    password_selectors = ["input[type='password']", "input[name='password']", "input[id*='password']"]
    submit_selectors = ["button[type='submit']", "button:has-text('Connexion')", "button:has-text('Se connecter')", "input[type='submit']"]

    email_sel = None
    for sel in email_selectors:
        if await page.locator(sel).count() > 0:
            email_sel = sel
            break
    password_sel = None
    for sel in password_selectors:
        if await page.locator(sel).count() > 0:
            password_sel = sel
            break
    submit_sel = None
    for sel in submit_selectors:
        if await page.locator(sel).count() > 0:
            submit_sel = sel
            break

    if not email_sel or not password_sel:
        raise RuntimeError("Champs de connexion non trouvés sur la page login.")

    await _human_type(page, email_sel, CONFIG["UNEMPLACEMENT_EMAIL"])
    await _human_mouse_move(page, n_moves=1)
    await _human_type(page, password_sel, CONFIG["UNEMPLACEMENT_PASSWORD"])
    await _human_mouse_move(page, n_moves=1)

    if submit_sel:
        await page.locator(submit_sel).click()
    else:
        await page.keyboard.press("Enter")

    # Attente d'un indicateur de connexion (dashboard ou URL changeante)
    await page.wait_for_load_state("networkidle")
    await asyncio.sleep(_human_delay(800, 1500))
    print(f"[INFO] Connecté — URL actuelle : {page.url}")


async def _scroll_ou_pagination(page: Page, captured_responses: List[Dict[str, Any]]) -> None:
    """Gère le scroll infini ou la pagination sur la page des leads."""
    max_scrolls = 30
    scrolls = 0
    last_height = 0

    while scrolls < max_scrolls:
        try:
            # 1. Scroll infini
            await page.mouse.wheel(0, random.randint(800, 1400))
            await asyncio.sleep(_human_delay(600, 1400))

            current_height = await page.evaluate("() => document.body.scrollHeight")
            if current_height == last_height:
                # 2. Bouton "Charger plus"
                bouton_plus = page.locator(SELECTEUR_BOUTON_CHARGER_PLUS).first
                if await bouton_plus.is_visible() and await bouton_plus.is_enabled():
                    await _human_mouse_move(page, n_moves=1)
                    await bouton_plus.click()
                    await asyncio.sleep(_human_delay(1200, 2200))
                    last_height = 0
                    scrolls += 1
                    continue
                # 3. Pagination classique
                bouton_next = page.locator(SELECTEUR_BOUTON_PAGINATION).first
                if await bouton_next.is_visible() and await bouton_next.is_enabled():
                    await _human_mouse_move(page, n_moves=1)
                    await bouton_next.click()
                    await asyncio.sleep(_human_delay(1200, 2200))
                    last_height = 0
                    scrolls += 1
                    continue
                break
            last_height = current_height
            scrolls += 1
        except Exception as e:
            print(f"[WARN] Problème lors du scroll/pagination : {e}")
            break


async def run_scraper() -> pd.DataFrame:
    """
    Fonction principale de scraping asynchrone.
    Retourne un DataFrame `df_demandes_clients` strictement typé.
    """
    captured_responses: List[Dict[str, Any]] = []

    async def on_response(response: Response) -> None:
        """Interception réseau : capture les réponses JSON/GraphQL probables."""
        try:
            content_type = (response.headers.get("content-type", "")).lower()
            url = response.url.lower()
            is_api = (
                "json" in content_type
                or "graphql" in url
                or "/api/" in url
                or "unemplacement" in url and "demand" in url
            )
            if not is_api:
                return
            body = await response.body()
            if not body:
                return
            try:
                payload = json.loads(body)
                captured_responses.append(payload)
                print(f"[NET] API interceptée : {response.url[:80]}...")
            except json.JSONDecodeError:
                pass
        except Exception as e:
            print(f"[WARN] Erreur interception réseau : {e}")

    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=CONFIG["HEADLESS"],
            slow_mo=CONFIG["SLOW_MO"],
            args=[
                "--disable-blink-features=AutomationControlled",
                "--no-sandbox",
                "--disable-dev-shm-usage"
            ]
        )
        context = await browser.new_context(
            user_agent=random.choice(USER_AGENTS),
            viewport={"width": 1920, "height": 1080},
            locale="fr-FR",
            timezone_id="Europe/Paris",
            extra_http_headers={
                "Accept-Language": "fr-FR,fr;q=0.9,en-US;q=0.8,en;q=0.7",
                "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8"
            }
        )

        page = await context.new_page()
        await _patch_webdriver(page)
        page.on("response", lambda response: asyncio.create_task(on_response(response)))

        try:
            await _login(page)

            # Navigation vers la page des demandes
            print(f"[INFO] Navigation vers {CONFIG['URL_LEADS']}")
            await page.goto(CONFIG["URL_LEADS"], wait_until="networkidle")
            await page.wait_for_load_state("networkidle")
            await asyncio.sleep(_human_delay(1000, 2000))

            # Scroll / pagination pour charger toutes les annonces
            await _scroll_ou_pagination(page, captured_responses)

            # Priorité au JSON réseau, sinon fallback HTML
            annonces = []
            if captured_responses:
                print(f"[INFO] {len(captured_responses)} réponses API interceptées — parsing JSON.")
                annonces = _extraire_annonces_json(captured_responses)

            if not annonces:
                print("[INFO] Aucun JSON exploitable — basculement parsing HTML.")
                annonces = await _extraire_annonces_html(page)

        except Exception as e:
            print(f"[ERROR] Erreur fatale scraping : {e}")
            annonces = []
        finally:
            await context.close()
            await browser.close()

    # Typage strict du DataFrame
    df = pd.DataFrame(annonces, columns=[
        "id_annonce", "ville", "code_postal", "surface_min", "surface_max",
        "budget_max_mensuel", "activite_detaillee", "besoin_extraction", "url_contact"
    ])

    dtype_map = {
        "id_annonce": "string",
        "ville": "string",
        "code_postal": "string",
        "surface_min": "Int64",
        "surface_max": "Int64",
        "budget_max_mensuel": "Float64",
        "activite_detaillee": "string",
        "besoin_extraction": "boolean",
        "url_contact": "string"
    }
    df = df.astype(dtype_map)
    df = df.fillna({"surface_min": 0, "surface_max": 0, "budget_max_mensuel": 0.0, "activite_detaillee": "", "url_contact": ""})

    print(f"[OK] {len(df)} annonces clients scrapées.")
    return df


# Exécution dans le notebook (asyncio déjà actif)
df_demandes_clients = await run_scraper()
df_demandes_clients.head()

## CELLULE 4 : INTÉGRATION DU CSV ET ALGORITHME DE CORRÉLATION (MATCHING)

In [ ]:
# --- CELLULE 4 : CSV PROFOODS + ALGORITHME DE MATCHING ---

import glob
import os

# Chemin vers le CSV source (export Google Sheet) ou le CSV déjà nettoyé
PATH_CSV_PROFOODS_SOURCE = os.environ.get(
    "PATH_CSV_PROFOODS_SOURCE",
    "/zpool/one/maxime.debaugnies/profoods/*.csv"
)
PATH_CSV_PROFOODS_NETTOYE = os.environ.get(
    "PATH_CSV_PROFOODS_NETTOYE",
    "/zpool/one/maxime.debaugnies/data/biens_profoods.csv"
)


def _detecter_csv_profoods_nettoye(path: str) -> bool:
    """Vérifie si le CSV est déjà au format standardisé."""
    try:
        with open(path, "r", encoding="utf-8") as f:
            header = f.readline().strip().lower()
        return "id_bien,adresse_ville,code_postal,surface_totale,loyer_cible,a_une_extraction" in header
    except Exception:
        return False


def _nettoyer_csv_profoods(source_path: str, output_path: str) -> pd.DataFrame:
    """
    Nettoie le CSV exporté du Google Sheet ProFoods (header multi-lignes)
    et génère un CSV standardisé avec les colonnes attendues par le notebook.
    """
    df = pd.read_csv(source_path, header=[1, 2])

    out_rows = []
    for idx, row in df.iterrows():
        ville = _normaliser_ville(row.get(("Ville", "Unnamed: 2_level_1"), ""))
        cp = _normaliser_code_postal(row.get(("Code postal", "Unnamed: 3_level_1"), ""))
        surface = _extraire_nombre(row.get(("Unnamed: 11_level_0", "Surface bâtie totale (m²)"), ""))
        loyer = _extraire_nombre(row.get(("Loyer attendu", "€HTHC/an"), ""))

        extraction_val = row.get(("Extraction", "Diamètre (mm)"), "")
        extraction = False
        if pd.notna(extraction_val):
            txt = str(extraction_val).strip()
            if txt not in {"", "/", "NA", "N/A", "na", "n/a"}:
                num = _extraire_nombre(txt)
                extraction = num is not None and num > 0

        adresse = str(row.get(("Adresse", "Unnamed: 5_level_1"), "")).strip() if pd.notna(row.get(("Adresse", "Unnamed: 5_level_1"), "")) else ""
        gmaps = str(row.get(("Lien GMAPS", "Unnamed: 6_level_1"), "")).strip() if pd.notna(row.get(("Lien GMAPS", "Unnamed: 6_level_1"), "")) else ""

        if not ville and not cp:
            continue

        out_rows.append({
            "id_bien": f"PF{idx:04d}",
            "adresse_ville": ville,
            "code_postal": cp,
            "surface_totale": int(surface) if surface else 0,
            "loyer_cible": loyer if loyer else 0.0,
            "a_une_extraction": extraction,
            "adresse_brute": adresse,
            "lien_gmaps": gmaps,
        })

    out_df = pd.DataFrame(out_rows)
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    out_df.to_csv(output_path, index=False)
    print(f"[OK] {len(out_df)} biens nettoyés écrits dans {output_path}")
    return out_df


def charger_csv_profoods(path: str) -> pd.DataFrame:
    """Charge et nettoie le CSV des biens ProFoods avec schéma strict."""
    # Si le fichier n'est pas encore nettoyé, chercher un source dans profoods/
    if not os.path.exists(path) or not _detecter_csv_profoods_nettoye(path):
        sources = glob.glob(PATH_CSV_PROFOODS_SOURCE)
        if sources:
            print(f"[INFO] CSV nettoyé non trouvé, nettoyage du source {sources[0]}...")
            return _nettoyer_csv_profoods(sources[0], path)
        else:
            raise FileNotFoundError(f"Aucun CSV ProFoods trouvé ni à {path} ni dans {PATH_CSV_PROFOODS_SOURCE}")

    df = pd.read_csv(path, dtype=str)

    # Mapping des colonnes attendues
    colonnes_requises = {
        "id_bien": "id_bien",
        "adresse_ville": "adresse_ville",
        "code_postal": "code_postal",
        "surface_totale": "surface_totale",
        "loyer_cible": "loyer_cible",
        "a_une_extraction": "a_une_extraction"
    }

    # Normalisation des noms de colonnes
    df.columns = [c.strip().lower().replace(" ", "_").replace("-", "_") for c in df.columns]

    # Renommage si présent
    for cible, nom in colonnes_requises.items():
        if nom not in df.columns:
            matches = [c for c in df.columns if cible in c]
            if matches:
                df.rename(columns={matches[0]: nom}, inplace=True)

    # Nettoyage
    df["adresse_ville"] = df["adresse_ville"].apply(_normaliser_ville)
    df["code_postal"] = df["code_postal"].apply(_normaliser_code_postal)
    df["surface_totale"] = df["surface_totale"].apply(lambda x: int(float(x)) if pd.notna(x) and str(x).strip() else 0)
    df["loyer_cible"] = df["loyer_cible"].apply(lambda x: float(x) if pd.notna(x) and str(x).strip() else 0.0)

    def _bool_extraction(v):
        if pd.isna(v):
            return False
        return str(v).strip().lower() in {"1", "true", "oui", "yes", "vrai"}

    df["a_une_extraction"] = df["a_une_extraction"].apply(_bool_extraction)

    df = df[["id_bien", "adresse_ville", "code_postal", "surface_totale", "loyer_cible", "a_une_extraction"]]

    dtype_map = {
        "id_bien": "string",
        "adresse_ville": "string",
        "code_postal": "string",
        "surface_totale": "Int64",
        "loyer_cible": "Float64",
        "a_une_extraction": "boolean"
    }
    df = df.astype(dtype_map)
    return df


def calculer_score(ligne_client: pd.Series, ligne_bien: pd.Series) -> int:
    """Calcule le score de matching 0-100 selon les règles métier."""
    score = 0

    # 1. Même ville/code postal
    if (ligne_client["code_postal"] and ligne_client["code_postal"] == ligne_bien["code_postal"]) or \
       (ligne_client["ville"] and ligne_client["ville"] == ligne_bien["adresse_ville"]):
        score += 50

    # 2. Surface
    surface_min = ligne_client["surface_min"] or 0
    surface_max = ligne_client["surface_max"] or 0
    surface_totale = ligne_bien["surface_totale"] or 0

    if surface_max > 0 and surface_min <= surface_totale <= surface_max:
        score += 30
    elif surface_max > 0 and surface_totale > surface_max:
        score -= 10

    # 3. Budget
    budget_max = ligne_client["budget_max_mensuel"] or 0.0
    loyer = ligne_bien["loyer_cible"] or 0.0
    if budget_max > 0 and loyer <= budget_max:
        score += 20

    # 4. Match technique bloquant
    if ligne_client["besoin_extraction"] and not ligne_bien["a_une_extraction"]:
        return 0

    return max(0, min(100, score))


def calculer_matches(df_clients: pd.DataFrame, df_biens: pd.DataFrame, score_min: int = 1) -> pd.DataFrame:
    """
    Produit toutes les combinaisons client × bien et retourne un DataFrame trié par score.
    """
    matches = []
    for _, client in df_clients.iterrows():
        for _, bien in df_biens.iterrows():
            try:
                score = calculer_score(client, bien)
                if score >= score_min:
                    matches.append({
                        "id_annonce": client["id_annonce"],
                        "ville_client": client["ville"],
                        "code_postal_client": client["code_postal"],
                        "surface_min_client": client["surface_min"],
                        "surface_max_client": client["surface_max"],
                        "budget_max_mensuel_client": client["budget_max_mensuel"],
                        "besoin_extraction_client": client["besoin_extraction"],
                        "url_contact": client["url_contact"],
                        "id_bien": bien["id_bien"],
                        "adresse_ville_bien": bien["adresse_ville"],
                        "code_postal_bien": bien["code_postal"],
                        "surface_totale_bien": bien["surface_totale"],
                        "loyer_cible_bien": bien["loyer_cible"],
                        "a_une_extraction_bien": bien["a_une_extraction"],
                        "score_matching": score
                    })
            except Exception as e:
                print(f"[WARN] Erreur matching {client['id_annonce']} × {bien['id_bien']} : {e}")

    df_matches = pd.DataFrame(matches)
    if df_matches.empty:
        return df_matches

    df_matches = df_matches.sort_values(
        ["score_matching", "id_annonce", "loyer_cible_bien"],
        ascending=[False, True, True]
    ).reset_index(drop=True)

    # Typage strict
    df_matches["score_matching"] = df_matches["score_matching"].astype("int64")
    df_matches["surface_min_client"] = df_matches["surface_min_client"].astype("Int64")
    df_matches["surface_max_client"] = df_matches["surface_max_client"].astype("Int64")
    df_matches["surface_totale_bien"] = df_matches["surface_totale_bien"].astype("Int64")
    df_matches["budget_max_mensuel_client"] = df_matches["budget_max_mensuel_client"].astype("Float64")
    df_matches["loyer_cible_bien"] = df_matches["loyer_cible_bien"].astype("Float64")
    df_matches["besoin_extraction_client"] = df_matches["besoin_extraction_client"].astype("boolean")
    df_matches["a_une_extraction_bien"] = df_matches["a_une_extraction_bien"].astype("boolean")

    return df_matches


# Chargement CSV (auto-nettoyage si nécessaire)
df_biens_profoods = charger_csv_profoods(CONFIG["PATH_CSV_PROFOODS"])
print(f"[OK] {len(df_biens_profoods)} biens ProFoods chargés.")

# Matching (score_min=1 pour exclure les 0, ou 0 pour tout conserver)
df_opportunites_matching = calculer_matches(df_demandes_clients, df_biens_profoods, score_min=1)
print(f"[OK] {len(df_opportunites_matching)} opportunités de matching générées.")
df_opportunites_matching.head()

## CELLULE 5 : SIMULATION D'EXPORT & STRUCTURATION SQL POUR VERCEL

In [ ]:
# --- CELLULE 5 : EXPORT LOCAL, INGESTION API & SCHÉMA SQL POUR VERCEL/SUPABASE ---

import os
import requests

# 1. Export local temporaire (même si le DataFrame est vide)
timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = "./data/outputs"
os.makedirs(output_dir, exist_ok=True)

csv_path = os.path.join(output_dir, f"profoods_opportunites_matching_{timestamp}.csv")
excel_path = os.path.join(output_dir, f"profoods_opportunites_matching_{timestamp}.xlsx")

if not df_opportunites_matching.empty:
    df_opportunites_matching.to_csv(csv_path, index=False)
    df_opportunites_matching.to_excel(excel_path, index=False, engine="openpyxl")
    print(f"[OK] Export CSV : {csv_path}")
    print(f"[OK] Export Excel : {excel_path}")
else:
    print("[INFO] Aucune opportunité à exporter — CSV/Excel non générés.")

# 2. Ingestion vers l'application ProKitchens (optionnel)
# Configurez PROFOODS_API_URL dans votre environnement, ex:
# export PROFOODS_API_URL="https://prokitchens-three.vercel.app/api/profoods/ingest"
PROFOODS_API_URL = os.environ.get(
    "PROFOODS_API_URL",
    "https://prokitchens-three.vercel.app/api/profoods/ingest"
)


def _dataframe_to_records(df: pd.DataFrame, columns: List[str]) -> List[Dict[str, Any]]:
    """Convertit un DataFrame en liste de dicts JSON-safe."""
    if df.empty:
        return []
    payload = df[columns].copy()
    payload = payload.where(pd.notna(payload), None)
    records = payload.to_dict(orient="records")
    # Conversion explicite des types numpy/pandas natifs
    for rec in records:
        for k, v in rec.items():
            if pd.isna(v):
                rec[k] = None
            elif isinstance(v, pd.Int64Dtype().type) or hasattr(v, "item"):
                rec[k] = int(v) if v is not None else None
            elif isinstance(v, float):
                rec[k] = float(v)
            elif isinstance(v, bool):
                rec[k] = bool(v)
            elif isinstance(v, str):
                rec[k] = str(v)
    return records


def pousser_vers_prokitchens(
    df_demandes: pd.DataFrame,
    df_biens: pd.DataFrame,
    api_url: str = PROFOODS_API_URL
) -> dict:
    """Pousse les DataFrames de demandes et de biens vers l'API ProKitchens."""
    demandes_columns = [
        "id_annonce", "ville", "code_postal", "surface_min", "surface_max",
        "budget_max_mensuel", "activite_detaillee", "besoin_extraction", "url_contact"
    ]
    biens_columns = [
        "id_bien", "adresse_ville", "code_postal", "surface_totale", "loyer_cible", "a_une_extraction"
    ]

    demandes_payload = _dataframe_to_records(df_demandes, demandes_columns)
    biens_payload = _dataframe_to_records(df_biens, biens_columns)

    print(f"[INFO] Push vers {api_url} : {len(demandes_payload)} demandes, {len(biens_payload)} biens")

    resp = requests.post(
        api_url,
        json={"demandes": demandes_payload, "biens": biens_payload},
        timeout=120,
    )
    resp.raise_for_status()
    return resp.json()


if os.environ.get("PUSH_TO_PROKITCHENS", "") == "1":
    try:
        result = pousser_vers_prokitchens(df_demandes_clients, df_biens_profoods, PROFOODS_API_URL)
        print(f"[OK] Données poussées vers ProKitchens : {result}")
    except Exception as e:
        print(f"[WARN] Échec de la poussée API : {e}")
else:
    print("[INFO] Set PUSH_TO_PROKITCHENS=1 pour pousser automatiquement vers l'API ProKitchens.")

# 3. Schéma SQL exact pour Postgres/Supabase (correspond aux tables créées dans l'app)
SCHEMA_SQL = """
-- Schéma SQL ProFoods pour Postgres/Supabase (application ProKitchens)

CREATE TABLE IF NOT EXISTS public.profoods_demandes_clients (
  id uuid PRIMARY KEY DEFAULT gen_random_uuid(),
  id_annonce text NOT NULL UNIQUE,
  ville text NOT NULL,
  code_postal text,
  surface_min integer NOT NULL DEFAULT 0,
  surface_max integer,
  budget_max_mensuel numeric(12,2),
  activite_detaillee text,
  besoin_extraction boolean NOT NULL DEFAULT false,
  url_contact text,
  scraped_at timestamptz NOT NULL DEFAULT now(),
  created_at timestamptz NOT NULL DEFAULT now(),
  updated_at timestamptz NOT NULL DEFAULT now()
);

CREATE INDEX IF NOT EXISTS idx_profoods_demandes_cp ON public.profoods_demandes_clients (code_postal);
CREATE INDEX IF NOT EXISTS idx_profoods_demandes_ville ON public.profoods_demandes_clients (ville);
CREATE INDEX IF NOT EXISTS idx_profoods_demandes_extraction ON public.profoods_demandes_clients (besoin_extraction);

CREATE TABLE IF NOT EXISTS public.profoods_biens (
  id uuid PRIMARY KEY DEFAULT gen_random_uuid(),
  id_bien text NOT NULL UNIQUE,
  adresse_ville text NOT NULL,
  code_postal text,
  surface_totale integer,
  loyer_cible numeric(12,2),
  a_une_extraction boolean NOT NULL DEFAULT false,
  imported_at timestamptz NOT NULL DEFAULT now(),
  created_at timestamptz NOT NULL DEFAULT now(),
  updated_at timestamptz NOT NULL DEFAULT now()
);

CREATE INDEX IF NOT EXISTS idx_profoods_biens_cp ON public.profoods_biens (code_postal);
CREATE INDEX IF NOT EXISTS idx_profoods_biens_ville ON public.profoods_biens (adresse_ville);
CREATE INDEX IF NOT EXISTS idx_profoods_biens_extraction ON public.profoods_biens (a_une_extraction);

CREATE TABLE IF NOT EXISTS public.profoods_opportunites (
  id uuid PRIMARY KEY DEFAULT gen_random_uuid(),
  demande_id uuid NOT NULL REFERENCES public.profoods_demandes_clients(id) ON DELETE CASCADE,
  bien_id uuid NOT NULL REFERENCES public.profoods_biens(id) ON DELETE CASCADE,
  score_matching integer NOT NULL CHECK (score_matching BETWEEN 0 AND 100),
  matched_at timestamptz NOT NULL DEFAULT now(),
  UNIQUE(demande_id, bien_id)
);

CREATE INDEX IF NOT EXISTS idx_profoods_opportunites_score ON public.profoods_opportunites (score_matching DESC);
CREATE INDEX IF NOT EXISTS idx_profoods_opportunites_demande ON public.profoods_opportunites (demande_id);
CREATE INDEX IF NOT EXISTS idx_profoods_opportunites_bien ON public.profoods_opportunites (bien_id);
"""

print(SCHEMA_SQL)